# Line Charts

A **line chart** connects data points with a continuous line, making it ideal
for showing trends over time or ordered categories. Use it when the sequence
of data points matters — for example, stock prices, temperature over days,
or any measurement that changes continuously.

**When to use:** time series, trends, continuous data
**When to avoid:** unordered categories, too many overlapping series

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

## Basic line chart with Matplotlib

The simplest form: a single series plotted against an index.
`plt.plot()` draws the line; `marker="o"` adds visible data points.

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
sales = [120, 145, 130, 160, 175, 190]

plt.figure(figsize=(8, 4))
plt.plot(months, sales, marker="o", linewidth=2)
plt.title("Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Sales ($)")
plt.tight_layout()
plt.show()

## Multi-series line chart with Seaborn

`sns.lineplot()` handles multiple series naturally via the `hue` parameter.
Seaborn also draws a confidence interval when multiple observations exist
per x-value — set `errorbar=None` to disable it.

In [ ]:
df = pd.DataFrame({
    "month": months * 2,
    "sales": sales + [100, 120, 115, 140, 155, 170],
    "region": ["North"] * 6 + ["South"] * 6,
})

plt.figure(figsize=(8, 4))
sns.lineplot(data=df, x="month", y="sales", hue="region", marker="o")
plt.title("Monthly Sales by Region")
plt.tight_layout()
plt.show()

# Scatter Charts

A **scatter chart** plots individual data points as dots on a two-dimensional
plane, with no line connecting them. Its primary purpose is to reveal the
**relationship (correlation) between two continuous variables** — whether it
is positive, negative, linear, curved, or absent altogether.

**When to use:** exploring correlations, spotting clusters or outliers,
comparing observed vs. predicted values
**When to avoid:** ordinal/categorical x-axis, more than a few thousand
points without transparency (`alpha`) or aggregation

## Basic scatter chart

`plt.scatter()` is the simplest entry point. Each dot represents one
observation. Jitter is not needed here because x and y are both continuous.
`alpha` controls transparency — useful when points overlap.

In [ ]:
rng = np.random.default_rng(42)
study_hours = rng.uniform(1, 10, 80)
exam_score = 40 + 5 * study_hours + rng.normal(0, 5, 80)

plt.figure(figsize=(7, 5))
plt.scatter(study_hours, exam_score, alpha=0.7, edgecolors="white", linewidths=0.4)
plt.title("Study Hours vs Exam Score")
plt.xlabel("Study Hours")
plt.ylabel("Exam Score")
plt.tight_layout()
plt.show()

## Scatter plot with linear approximation

`np.polyfit(x, y, deg=1)` fits a polynomial of degree 1 (a straight line)
using least squares. The resulting coefficients are passed to `np.poly1d`
to evaluate the line at any x. This overlay makes the trend immediately
visible even when the raw scatter is noisy.

In [ ]:
coeffs = np.polyfit(study_hours, exam_score, deg=1)
line_fn = np.poly1d(coeffs)
x_range = np.linspace(study_hours.min(), study_hours.max(), 200)

plt.figure(figsize=(7, 5))
plt.scatter(study_hours, exam_score, alpha=0.7, edgecolors="white", linewidths=0.4, label="Observations")
plt.plot(x_range, line_fn(x_range), color="tomato", linewidth=2, label=f"Linear fit  y = {coeffs[0]:.2f}x + {coeffs[1]:.2f}")
plt.title("Study Hours vs Exam Score — Linear Fit")
plt.xlabel("Study Hours")
plt.ylabel("Exam Score")
plt.legend()
plt.tight_layout()
plt.show()

## Scatter plot with quadratic (square function) approximation

When the relationship curves — here, diminishing returns: initial study time
helps a lot, extra hours help less — a degree-2 polynomial fits better than
a line. The label shows the full equation so readers can judge the curve's
shape at a glance.

In [ ]:
rng2 = np.random.default_rng(7)
hours2 = rng2.uniform(0, 10, 100)
score2 = 30 + 12 * hours2 - 0.9 * hours2**2 + rng2.normal(0, 4, 100)

coeffs2 = np.polyfit(hours2, score2, deg=2)
curve_fn = np.poly1d(coeffs2)
x_range2 = np.linspace(hours2.min(), hours2.max(), 200)

a, b, c = coeffs2
plt.figure(figsize=(7, 5))
plt.scatter(hours2, score2, alpha=0.7, edgecolors="white", linewidths=0.4, label="Observations")
plt.plot(x_range2, curve_fn(x_range2), color="seagreen", linewidth=2,
         label=f"Quadratic fit  y = {a:.2f}x² + {b:.2f}x + {c:.2f}")
plt.title("Study Hours vs Exam Score — Quadratic Fit")
plt.xlabel("Study Hours")
plt.ylabel("Exam Score")
plt.legend()
plt.tight_layout()
plt.show()

## Scatter plot: actual vs predicted values

A standard diagnostic plot in predictive modelling. Each point is one
observation; x is the model's prediction, y is the ground truth.
A **perfect model** would place all points on the diagonal (predicted = actual).
The dashed 45° line — drawn from the joint min to the joint max — makes
deviations from perfection immediately visible: points above the line mean
the model under-predicted; points below mean it over-predicted.

In [ ]:
rng3 = np.random.default_rng(99)
actual = rng3.uniform(20, 100, 120)
predicted = actual + rng3.normal(0, 8, 120)

axis_min = min(actual.min(), predicted.min()) - 2
axis_max = max(actual.max(), predicted.max()) + 2

plt.figure(figsize=(6, 6))
plt.scatter(predicted, actual, alpha=0.6, edgecolors="white", linewidths=0.4, label="Observations")
plt.plot([axis_min, axis_max], [axis_min, axis_max],
         linestyle="--", color="tomato", linewidth=1.5, label="Perfect prediction (45°)")
plt.xlim(axis_min, axis_max)
plt.ylim(axis_min, axis_max)
plt.title("Actual vs Predicted Values")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.legend()
plt.tight_layout()
plt.show()

## Actual vs Predicted — real dataset, large scale

We now use the **California Housing** dataset (built into scikit-learn,
20 640 observations) with a plain `LinearRegression` model.
The target variable is median house value (in $100 000s) per census block.

Steps: load → train/test split → fit → predict on the test split (~5 000 samples).

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

### Scatter plot — overcrowding problem

With ~5 000 points the scatter plot already suffers from severe **overplotting**:
dots pile on top of each other in the dense central region, making it
impossible to tell whether there are 10 or 10 000 points in any given area.
Lowering `alpha` helps at the edges but the core remains a solid blob — we
cannot reliably read the density distribution from this chart alone.

In [ ]:
diag_min = min(y_test.min(), y_pred.min()) - 0.1
diag_max = max(y_test.max(), y_pred.max()) + 0.1

plt.figure(figsize=(6, 6))
plt.scatter(y_pred, y_test, alpha=0.15, s=8, edgecolors="none")
plt.plot([diag_min, diag_max], [diag_min, diag_max],
         linestyle="--", color="tomato", linewidth=1.5, label="Perfect prediction (45°)")
plt.xlim(diag_min, diag_max)
plt.ylim(diag_min, diag_max)
plt.title("Actual vs Predicted — California Housing (scatter)")
plt.xlabel("Predicted value ($100k)")
plt.ylabel("Actual value ($100k)")
plt.legend()
plt.tight_layout()
plt.show()

### Hexbin plot — density-aware alternative

A **hexbin plot** tessellates the plane with hexagons and colours each cell
by the count of points it contains. This gives a true picture of the
**joint density**: dark hexagons are regions where many predictions land,
light ones are sparse — something a scatter plot cannot convey.

Key parameters:
- `gridsize` — hexagons along the x-axis; larger = finer resolution
- `cmap` — use a sequential map (`YlOrRd`) so low- and high-density cells
  remain visually distinct
- `mincnt=1` — hide empty hexagons entirely instead of colouring them zero
- `colorbar` — mandatory; without it the colour encoding is uninterpretable

The 45° dashed line is preserved so model accuracy is still readable
alongside the density information.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
hb = ax.hexbin(y_pred, y_test, gridsize=20, cmap="YlOrRd", mincnt=1)
ax.plot([diag_min, diag_max], [diag_min, diag_max],
        linestyle="--", color="steelblue", linewidth=1.5, label="Perfect prediction (45°)")
ax.set_xlim(diag_min, diag_max)
ax.set_ylim(diag_min, diag_max)
ax.set_title("Actual vs Predicted — California Housing (hexbin)")
ax.set_xlabel("Predicted value ($100k)")
ax.set_ylabel("Actual value ($100k)")
ax.legend()
fig.colorbar(hb, ax=ax, label="Number of observations")
plt.tight_layout()
plt.show()